# 05 — scRNA-seq with Seurat

**Run on: x86** — Requires Cell Ranger output. Run `scripts/python/download_cellranger_data.py` on x86, then `scripts/python/run_cellranger.py` or `cellranger count` → filtered count matrices.

**Target:** Figs 1, 2, 3, 5 — UMAP, clusters, DE, stress correlation

## Datasets

- **Fig 1:** Sham, TAC, TAC_JQ1 non-CM
- **Fig 2:** CD45+ Sham, TAC, TAC_Brd4KO
- **Fig 3:** snRNA TAC vs TAC_Brd4KO (whole heart)
- **Fig 5:** TAC IgG vs TAC anti-IL1B

## Pipeline

1. Read10X / CreateSeuratObject
2. QC: nFeature 2000–7500, nCount < 80k, mito < 15%
3. SCTransform → Harmony → UMAP → clusters
4. FindMarkers for DE

In [1]:
# Setup (R)
suppressPackageStartupMessages({
  library(Seurat)
  library(ggplot2)
})

set.seed(1)

out_base <- file.path("..", "output", "cellranger")

# Minimal Fig 1 scRNA reps (one each): Sham / TAC / TAC_JQ1
runs_fig1_min <- c(
  Sham = "SRR22882168",
  TAC = "SRR22882171",
  TAC_JQ1 = "SRR22882174"
)

# Helper: find a 10x matrix path for a run id
find_10x_dir <- function(run_id, base_dir = out_base) {
  candidates <- c(
    file.path(base_dir, run_id, "outs", "filtered_feature_bc_matrix"),
    file.path(base_dir, run_id, "outs", "filtered_feature_bc_matrix", ""),
    file.path(base_dir, run_id, "outs", "filtered_feature_bc_matrix.h5")
  )
  for (p in candidates) {
    if (dir.exists(p) || file.exists(p)) return(p)
  }
  NA_character_
}

paths <- vapply(unname(runs_fig1_min), find_10x_dir, character(1))
print(data.frame(condition = names(runs_fig1_min), run_id = unname(runs_fig1_min), path = paths))

if (all(is.na(paths))) {
  stop(
    "No Cell Ranger outputs found under ", out_base, "\n\n",
    "Expected: output/cellranger/<SRR>/outs/filtered_feature_bc_matrix (or .h5)\n",
    "Run (on x86): python scripts/python/run_cellranger.py --ref-scrna ... --ref-atac ...\n",
    "Then re-run this notebook."
  )
}

            condition      run_id path
SRR22882168      Sham SRR22882168 <NA>
SRR22882171       TAC SRR22882171 <NA>
SRR22882174   TAC_JQ1 SRR22882174 <NA>


ERROR: Error: No Cell Ranger outputs found under ../output/cellranger

Expected: output/cellranger/<SRR>/outs/filtered_feature_bc_matrix (or .h5)
Run (on x86): python scripts/python/run_cellranger.py --ref-scrna ... --ref-atac ...
Then re-run this notebook.


In [ ]:
# Load Fig 1 minimal runs (if available)

read_10x_any <- function(path) {
  if (is.na(path)) return(NULL)
  if (dir.exists(path)) {
    Read10X(data.dir = path)
  } else if (file.exists(path) && grepl("\\.h5$", path)) {
    Read10X_h5(filename = path)
  } else {
    NULL
  }
}

objs <- list()
for (cond in names(runs_fig1_min)) {
  rid <- runs_fig1_min[[cond]]
  p <- find_10x_dir(rid)
  if (is.na(p)) {
    message("Missing: ", rid, " (", cond, ")")
    next
  }
  m <- read_10x_any(p)
  if (is.null(m)) {
    message("Unreadable matrix for: ", rid, " at ", p)
    next
  }
  s <- CreateSeuratObject(counts = m, project = "Alexanian_scRNA", min.cells = 3, min.features = 200)
  s$condition <- cond
  s$run_id <- rid
  s$orig.ident <- paste0(cond, "_", rid)
  objs[[cond]] <- s
}

if (length(objs) == 0) stop("No readable matrices found.")

obj <- if (length(objs) == 1) {
  objs[[1]]
} else {
  # Merge all; add.cell.ids helps keep barcodes unique
  merge(x = objs[[1]], y = objs[-1], add.cell.ids = names(objs))
}

obj

In [ ]:
# QC (minimal + slide-friendly)

obj[["percent.mt"]] <- PercentageFeatureSet(obj, pattern = "^mt-")

VlnPlot(obj, features = c("nFeature_RNA", "nCount_RNA", "percent.mt"), group.by = "condition", ncol = 3)

# Light-touch filters (tune if needed)
obj <- subset(obj, subset = nFeature_RNA >= 200 & nFeature_RNA <= 7500 & nCount_RNA <= 80000 & percent.mt <= 15)
obj

In [ ]:
# Normalize + UMAP + clustering

obj <- NormalizeData(obj)
obj <- FindVariableFeatures(obj, selection.method = "vst", nfeatures = 2000)
obj <- ScaleData(obj)
obj <- RunPCA(obj, npcs = 30, verbose = FALSE)
obj <- FindNeighbors(obj, dims = 1:30)
obj <- FindClusters(obj, resolution = 0.5)
obj <- RunUMAP(obj, dims = 1:30)

p1 <- DimPlot(obj, reduction = "umap", group.by = "seurat_clusters", label = TRUE) + ggtitle("Clusters")
p2 <- DimPlot(obj, reduction = "umap", group.by = "condition") + ggtitle("Condition")

p1 + p2

In [ ]:
# Quick marker sanity check (pick a small set)

# A few canonical markers for broad cell types in mouse heart
markers <- list(
  Fibroblast = c("Col1a1", "Col1a2", "Dcn"),
  Immune = c("Ptprc"),
  Endothelial = c("Pecam1", "Kdr"),
  Myeloid = c("Lyz2", "Adgre1"),
  Tcell = c("Cd3d", "Trac"),
  Bcell = c("Ms4a1", "Cd79a"),
  Cardiomyocyte = c("Tnnt2", "Actc1")
)

DotPlot(obj, features = unique(unlist(markers))) + RotatedAxis()

# Optional: identify top markers per cluster
markers_tbl <- FindAllMarkers(obj, only.pos = TRUE, min.pct = 0.25, logfc.threshold = 0.25)
head(markers_tbl[order(markers_tbl$p_val_adj, -markers_tbl$avg_log2FC), ], 20)

## Practical tips (Cell Ranger + Seurat)

**1) Prerequisite check before running Seurat**

This notebook expects one of:

- `output/cellranger/<SRR>/outs/filtered_feature_bc_matrix/` (MEX folder), or
- `output/cellranger/<SRR>/outs/filtered_feature_bc_matrix.h5`

for each SRR used in your selected figure subset.

**2) Cell Ranger v10 CLI gotchas**

When running `cellranger count`, v10 expects explicit boolean args such as:

- `--include-introns true|false`
- `--create-bam true|false`

Use the project wrapper script so these flags are handled consistently.

**3) Minimum dataset for a quick Fig 1-style UMAP**

- Sham: `SRR22882168`
- TAC: `SRR22882171`
- TAC_JQ1: `SRR22882174`

This is enough for a minimal presentation plot.

**4) QC interpretation**

- High `nFeature_RNA` / `nCount_RNA` outliers may be doublets.
- High `percent.mt` often indicates stressed/low-quality cells.
- Keep QC thresholds simple and report them explicitly in slides.

**5) Talk-friendly wording**

"After standard Seurat QC and dimensionality reduction, we compare cluster structure and condition composition. Marker expression supports broad cell identity labels, and condition shifts are interpreted as hypothesis-generating unless supported by formal differential testing and replicate-aware statistics."

**6) Common troubleshooting**

- **No matrices found** -> Cell Ranger outputs missing or wrong path.
- **Barcode collisions after merge** -> keep `add.cell.ids` (already used).
- **Very slow runtime** -> lower PCs (e.g., 20) and skip `FindAllMarkers` on first pass.